## Verify the retail deployment

These checks confirm that:
- Every Bronze row is accounted for in Silver or quarantine.
- Row counts and Gold totals match our validated, unchanged source file.

The target_schema parameter selects which environment to check.
A failed assertion stops the notebook and fails its job task.

These expected values apply to this fixed dataset. They must be reviewed
if we intentionally add new source files.

In [0]:
%python
from decimal import Decimal

# Default to dev when running manually.
# The job will supply the schema for its deployment target.
dbutils.widgets.text("target_schema", "workspace.retail_demo4_dev")
target_schema = dbutils.widgets.get("target_schema")

print(f"Checking: {target_schema}")

Checking: workspace.retail_demo4_dev


In [0]:
%python
# Count each layer to confirm no rows disappeared during processing.
bronze_rows = spark.table(f"{target_schema}.bronze_online_retail").count()
silver_rows = spark.table(f"{target_schema}.silver_online_retail").count()
quarantine_rows = spark.table(f"{target_schema}.quarantine_online_retail").count()

print(f"Bronze: {bronze_rows:,}")
print(f"Silver: {silver_rows:,}")
print(f"Quarantine: {quarantine_rows:,}")

# An assertion raises an error if its condition is false.
assert bronze_rows == silver_rows + quarantine_rows, "Some rows are unaccounted for."

# Check the expected results for our unchanged source file.
assert bronze_rows == 541909, "Unexpected Bronze count."
assert silver_rows == 539392, "Unexpected Silver count."
assert quarantine_rows == 2517, "Unexpected quarantine count."

print("Row count checks passed.")

Bronze: 541,909
Silver: 539,392
Quarantine: 2,517
Row count checks passed.


In [0]:
%python
# SQL calculates the summary; first() retrieves its single result row.
totals = spark.sql(f"""
    SELECT
        COUNT(*) AS sales_days,
        SUM(sales_amount) AS sales_amount,
        SUM(cancellation_amount) AS cancellation_amount,
        SUM(net_amount) AS net_amount
    FROM {target_schema}.gold_daily_sales
""").first()

# Decimal preserves exact decimal values for these comparisons.
assert totals.sales_days == 305, "Unexpected number of sales days."
assert totals.sales_amount == Decimal("10666684.5440"), "Sales total differs."
assert totals.cancellation_amount == Decimal("-896812.4900"), "Cancellation total differs."
assert totals.net_amount == Decimal("9769872.0540"), "Net total differs."

print(f"Net amount: {totals.net_amount}")
print("Gold total checks passed.")

Net amount: 9769872.0540
Gold total checks passed.
